# Sentinel-2 matching experiment

This notebook builds the pair manifest and compares SIFT, ORB, and LoFTR on the same Sentinel-2 scene pair. Every method prints progress and saves metrics and visualizations.

## 1. Configuration

The parameters below control the input datasets, RGB bands, image scaling, tile geometry, and optional methods.

In [ ]:
from pathlib import Path
import shutil
import importlib.util

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/matcher_outputs')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
BANDS = 'B04,B03,B02'
MAX_SIDE = 1600
TILE_SIZE = 1024
OVERLAP = 128
LEFT_PATH = None
RIGHT_PATH = None
RUN_SIFT = True
RUN_ORB = True
RUN_LOFTR = True

_candidates = sorted({path.parent for path in INPUT_ROOT.rglob('match_images.py') if path.parent.name == 'sentinel_matching'}, key=str) if INPUT_ROOT.is_dir() else []
MATCHER_DIR = _candidates[0] if _candidates else None
MATCHER_READY = MATCHER_DIR is not None
if MATCHER_READY:
    TARGET_DIR = Path('/kaggle/working/src/sentinel_matching')
    if MATCHER_DIR.resolve() != TARGET_DIR.resolve():
        TARGET_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(MATCHER_DIR, TARGET_DIR, dirs_exist_ok=True)
    print(f'code_dir={TARGET_DIR}')
else:
    print('Matcher code was not found in /kaggle/input.')


## 2. Creation of dataset: code and dependencies

The archive is copied to `/kaggle/working/src/sentinel_matching` so every project script is called through the same `!python src/sentinel_matching/...` interface. Dependencies are installed from the repository requirements file.

In [ ]:
if MATCHER_READY:
    !pip install -q -r /kaggle/working/src/sentinel_matching/requirements.txt
    print('dependencies_ready=True')
else:
    print('dependency_installation_skipped=True')

## 3. Creation of dataset: scene discovery and pair selection

The notebook searches for Sentinel-2 `.SAFE` scenes, groups them by tile ID, and selects two dates from the same tile when possible. Manual `LEFT_PATH` and `RIGHT_PATH` override automatic selection.

In [ ]:
def scene_candidates(root):
    return sorted({path.resolve() for path in root.rglob('*.SAFE') if path.is_dir()}, key=str)

SAFE_SCENES = scene_candidates(INPUT_ROOT)
print(f'safe_scenes={len(SAFE_SCENES)}')
for scene in SAFE_SCENES[:10]:
    print(scene)

def choose_pair():
    if LEFT_PATH and RIGHT_PATH and Path(LEFT_PATH).exists() and Path(RIGHT_PATH).exists():
        return str(Path(LEFT_PATH).resolve()), str(Path(RIGHT_PATH).resolve())
    groups = {}
    for scene in SAFE_SCENES:
        match = re.search(r'_T(\d{2}[A-Z]{3})_', scene.name.upper())
        groups.setdefault(match.group(1) if match else 'unknown', []).append(scene)
    for scenes in groups.values():
        if len(scenes) >= 2:
            return str(scenes[0]), str(scenes[1])
    return (str(SAFE_SCENES[0]), str(SAFE_SCENES[1])) if len(SAFE_SCENES) >= 2 else (None, None)

LEFT, RIGHT = choose_pair()
PAIR_READY = LEFT is not None and RIGHT is not None
print(f'left={LEFT}')
print(f'right={RIGHT}')

## 4. Creation of dataset: pair manifest

This step creates same-tile positive pairs and different-tile negative pairs. The script shows progress while scanning SAFE scenes and prints the number of positive and negative pairs. With fewer than three geographic tile groups it uses a clearly marked pair-level fallback, because a leakage-safe three-way split is impossible.

In [ ]:
if MATCHER_READY and SAFE_SCENES:
    print('pair_manifest_mode=smoke_test_pair_level_fallback')
    !python src/sentinel_matching/pair_manifest.py --root /kaggle/input --output /kaggle/working/matcher_outputs/pairs.csv --allow-pair-level-fallback
else:
    print('pair_manifest_skipped=True')

## 5. SIFT

SIFT is the scale- and rotation-tolerant classical baseline. It uses local descriptors, Lowe-style filtering, and RANSAC geometric verification. The output includes keypoints, candidate matches, inliers, inlier ratio, runtime, CSV metrics, and the strongest tile visualization.

In [ ]:
if MATCHER_READY and PAIR_READY and RUN_SIFT:
    !python src/sentinel_matching/benchmark_matching.py --left "{LEFT}" --right "{RIGHT}" --bands "{BANDS}" --methods sift --tile-size {TILE_SIZE} --overlap {OVERLAP} --device auto --output-dir /kaggle/working/matcher_outputs/sift
else:
    print('sift_skipped=True')

## 6. ORB

ORB is the faster binary-descriptor baseline. It is useful for comparing speed and robustness against SIFT under the same tile size, overlap, bands, and RANSAC settings.

In [ ]:
if MATCHER_READY and PAIR_READY and RUN_ORB:
    !python src/sentinel_matching/benchmark_matching.py --left "{LEFT}" --right "{RIGHT}" --bands "{BANDS}" --methods orb --tile-size {TILE_SIZE} --overlap {OVERLAP} --device auto --output-dir /kaggle/working/matcher_outputs/orb
else:
    print('orb_skipped=True')

## 7. LoFTR

LoFTR is a pretrained deep local matcher. It predicts correspondences without hand-crafted descriptors and then uses RANSAC to count geometrically consistent inliers. The progress bar is shown per image tile; pretrained weights may be downloaded on the first run.

In [ ]:
if MATCHER_READY and PAIR_READY and RUN_LOFTR:
    if importlib.util.find_spec('kornia') is not None and importlib.util.find_spec('torch') is not None:
        !python src/sentinel_matching/benchmark_matching.py --left "{LEFT}" --right "{RIGHT}" --bands "{BANDS}" --methods loftr --tile-size {TILE_SIZE} --overlap {OVERLAP} --device auto --output-dir /kaggle/working/matcher_outputs/loftr
    else:
        print('loftr_skipped_missing_kornia_or_torch=True')
else:
    print('loftr_skipped=True')

## 8. Metrics and logs

Every method writes a `benchmark.csv`. This cell combines all available CSV files and displays the comparison table together with the generated visualizations.

In [ ]:
from IPython.display import Image, display
import pandas as pd

csv_files = sorted(WORK_ROOT.rglob('benchmark.csv'))
tables = []
for csv_path in csv_files:
    table = pd.read_csv(csv_path)
    table.insert(0, 'source', str(csv_path.parent.name))
    tables.append(table)
    print(f'loaded_metrics={csv_path}')
if tables:
    metrics = pd.concat(tables, ignore_index=True)
    display(metrics)
    metrics.to_csv(WORK_ROOT / 'all_methods_metrics.csv', index=False)
    print(f'saved_metrics={WORK_ROOT / "all_methods_metrics.csv"}')
else:
    print('no_benchmark_csv_found=True')

for image_path in sorted(WORK_ROOT.rglob('*.png')):
    print(f'visualization={image_path}')
    display(Image(filename=str(image_path)))

In [ ]:
import os
import zipfile
from pathlib import Path

import wandb
from tqdm.auto import tqdm


WORK_ROOT = Path("/kaggle/working")
ZIP_PATH = WORK_ROOT / "kaggle_working_snapshot.zip"

WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "image-matching")
WANDB_RUN_NAME = os.environ.get("WANDB_RUN_NAME", "matching-upload")



if not WANDB_API_KEY or not WANDB_ENTITY:
    raise ValueError("Set WANDB_API_KEY and WANDB_ENTITY in the environment before uploading.")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY, relogin=True)

ZIP_PATH.unlink(missing_ok=True)

files = [
    path
    for path in WORK_ROOT.rglob("*")
    if path.is_file() and path.resolve() != ZIP_PATH.resolve()
]

with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for file_path in tqdm(files, desc="Creating ZIP"):
        archive_name = Path("kaggle_working") / file_path.relative_to(WORK_ROOT)
        archive.write(file_path, arcname=str(archive_name))

print(f"Created: {ZIP_PATH}")
print(f"Files packed: {len(files)}")
print(f"Archive size: {ZIP_PATH.stat().st_size / 1024**2:.2f} MB")


with wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    job_type="upload-working-directory",
) as run:
    artifact = wandb.Artifact(
        name="kaggle-working-snapshot",
        type="experiment",
        metadata={
            "source_directory": str(WORK_ROOT),
            "file_count": len(files),
            "archive_name": ZIP_PATH.name,
        },
    )

    artifact.add_file(
        local_path=str(ZIP_PATH),
        name=ZIP_PATH.name,
    )

    run.log_artifact(artifact)
    artifact.wait()

    print(f"Uploaded artifact: {artifact.name}")

In [ ]:
# Recreate reproducible benchmark charts from the aggregate CSV.
!python src/sentinel_matching/utils/plot_benchmark.py --input "{WORK_ROOT / 'matcher_outputs' / 'all_methods_metrics.csv'}" --output-dir "{WORK_ROOT / 'matcher_outputs' / 'charts'}"